# DMRC Contract Intelligence — Production Deployment
## Notebook 03 · Production Deployment of the DMRC Enterprise RAG System

---

**Project:** Enterprise Retrieval-Augmented Generation platform for DMRC (Delhi Metro Rail Corporation) contract intelligence.

**Scope of this notebook:** deployment and production serving *only*. This notebook clones and runs the already-completed `dmrc_deploy` repository exactly as committed — it does not re-implement, redesign, or modify any retrieval, reranking, prompt-engineering, or generation logic. Every step below is running code that already exists in `src/app.py`, `src/hybrid_retriever.py`, `src/reranker.py`, `src/prompt_engineering.py`, and `src/gemma_inference.py`.

**Companion notebooks:**
- `01_Setup_and_Retrieval_Validation.ipynb` — validates the non-LLM retrieval stack (dense / BM25 / hybrid / reranking) against the production ChromaDB collection.
- `02_Gemma_Inference_and_Serving.ipynb` — validates Gemma-2-9B-it loading, generation, and the FastAPI `/ask` pipeline in-notebook.

This notebook assumes both of the above have already passed validation. It does **not** repeat their retrieval-quality or generation-quality checks — it takes the validated pipeline and deploys it as a publicly reachable, end-to-end system (backend API + frontend UI) suitable for a live demonstration.

**Runtime requirement:** GPU runtime (Colab: *Runtime → Change runtime type → GPU*). An A100/L4 with ≥24GB VRAM runs bf16 Gemma-2-9B directly; a smaller GPU (T4) should set `GEMMA_USE_4BIT=1` in Section 5.

## 1. Introduction

**Purpose of this notebook**
Deploy the backend of the completed DMRC Contract Intelligence RAG system — from a clean runtime to a publicly reachable API — and validate it end to end. This is the "demo day" notebook: run it top to bottom and you have a live, publicly callable backend, ready for a locally-run frontend or any other HTTP client.

**Deployment workflow**
This notebook proceeds through six operational stages, in order:

1. **Environment setup** — clone the repository, install dependencies, confirm GPU/runtime.
2. **Hugging Face authentication** — required for the gated `google/gemma-2-9b-it` checkpoint.
3. **Model preparation** — load and smoke-test the production model (tokenizer, quantization, GPU placement), then release notebook-side GPU memory.
4. **Backend deployment** — start the FastAPI application (`src/app.py`) as a background process.
5. **Cloudflare Tunnel** — expose the local backend on a public HTTPS URL.
6. **End-to-end testing, validation & troubleshooting** — exercise the full pipeline through the public API with a structured checklist plus fixes for common failure modes.

**Scope note:** this notebook covers **backend deployment only**. The React frontend (`frontend/`) is run locally, outside Colab, and is out of scope here — this notebook's only deliverable is a live, publicly reachable FastAPI backend that a locally-run frontend (or any HTTP client) can call.

**Overall architecture**

```
┌─────────────────────┐        HTTPS         ┌─────────────────────┐
│   Any HTTP client /  │ ───────────────────▶ │   Cloudflare Tunnel   │
│   locally-run React  │ ◀─────────────────── │   (backend :8000)     │
│   frontend (outside  │                        └───────────┬─────────┘
│   this notebook)      │                                    │
└─────────────────────┘                          (this Colab VM)
                                                  ┌──────────▼──────────┐
                                                  │   FastAPI (app.py)   │
                                                  │   POST /ask          │
                                                  │   GET  /status, /     │
                                                  └──────────┬──────────┘
                                                             │
                                ┌────────────────────────────┼────────────────────────────┐
                                ▼                             ▼                             ▼
                       ┌────────────────┐          ┌──────────────────┐          ┌──────────────────┐
                       │  Hybrid Search  │          │  Cross-Encoder     │          │  Gemma-2-9B-it     │
                       │  (dense BGE-M3   │ ───────▶ │  Reranker           │ ───────▶ │  (bf16 / 4-bit)     │
                       │   + BM25)         │          │  (BGE-reranker-v2) │          │  grounded generation │
                       └────────┬─────────┘          └──────────────────┘          └──────────────────┘
                                │
                       ┌────────▼────────┐
                       │   ChromaDB        │
                       │   (chroma_db/,     │
                       │   353 vectors)     │
                       └──────────────────┘
```

**What will be deployed**
- The FastAPI backend at `src/app.py`, unmodified, serving `GET /`, `GET /status`, and `POST /ask` over `uvicorn`.
- The pre-built, repository-shipped ChromaDB collection (`chroma_db/`) — no re-ingestion or re-embedding.
- `google/gemma-2-9b-it`, `BAAI/bge-m3`, and `BAAI/bge-reranker-v2-m3`, loaded exactly as `src/gemma_inference.py`, `src/query.py`, and `src/reranker.py` already implement.
- One Cloudflare Quick Tunnel, giving the backend a public HTTPS URL for the duration of this Colab session — ready for a locally-run frontend to call via `VITE_API_URL`.

## 2. Environment Setup

**Purpose**
Obtain a clean, current checkout of the `dmrc_deploy` repository, install its exact dependencies, and confirm the runtime (GPU, Python version) is suitable for production serving.

**Explanation**
This mirrors the clone/install/restart pattern used in Notebooks 01 and 02: dependencies are installed **before** any package that depends on `numpy`'s C ABI (`torch`, `transformers`, etc.) is imported, and the kernel is restarted immediately afterward so those packages load cleanly against the freshly-installed `numpy` — skipping this restart is what causes a `numpy.dtype size changed, may indicate binary incompatibility` error later in the notebook. This notebook always re-clones fresh (rather than `git pull`-ing an existing checkout) so a demo run is guaranteed to be running the current `main` branch, not a stale local copy left over from a previous session.

In [ ]:
import sys, platform

print("=" * 70)
print("  RUNTIME VERIFICATION")
print("=" * 70)
print(f"Python version : {sys.version.split()[0]}")
print(f"Platform       : {platform.platform()}")

try:
    import google.colab  # noqa: F401
    RUNNING_IN_COLAB = True
    print("Environment    : Google Colab")
except ImportError:
    RUNNING_IN_COLAB = False
    print("Environment    : Local / non-Colab Jupyter")


### Repository cloning and working directory setup

Clones the latest `main` branch fresh. If a previous clone exists in this Colab session it is removed first, so re-running this notebook always deploys the current `HEAD` — nothing here is cached or reused across runs.

In [ ]:
import os, shutil

REPO_URL = "https://github.com/sunvantaconsultancysolutions-design/dmrc_deploy.git"
REPO_DIR = "/content/dmrc_deploy"

if os.path.exists(REPO_DIR):
    print(f"Removing existing clone at {REPO_DIR} to re-clone the latest version...")
    shutil.rmtree(REPO_DIR)

clone_status = os.system(f"git clone --depth 1 {REPO_URL} {REPO_DIR}")

if clone_status == 0 and os.path.isdir(REPO_DIR):
    print(f"\n\u2705 SUCCESS: Cloned latest repo into {REPO_DIR}")
else:
    raise RuntimeError("\u274c FAILED: git clone did not succeed -- check network access and the REPO_URL above.")

get_ipython().run_line_magic("cd", REPO_DIR)
get_ipython().system("git log -1 --oneline")


**Expected Output**
`✅ SUCCESS: Cloned latest repo into /content/dmrc_deploy`, the working directory switched to the repo root, and the single-line summary of the commit currently deployed — so results here are traceable back to an exact repository version.

### Dependency installation

Installs straight from the repository's own `requirements.txt` — nothing rewritten by hand. `bitsandbytes` / `nvidia-nvjitlink-cu13` (needed only for the 4-bit path) are installed best-effort, matching the exact fallback the repository's own `Dockerfile` uses, so a wheel failure there does not block the bf16 path.

In [ ]:
# Core dependencies, straight from the repo's requirements.txt.
core_status = os.system(
    'grep -v -E "^(bitsandbytes|nvidia-nvjitlink-cu13)" requirements.txt > /tmp/requirements_core.txt '
    "&& pip install -q -r /tmp/requirements_core.txt"
)
if core_status == 0:
    print("\u2705 SUCCESS: Core dependencies installed from requirements.txt")
else:
    raise RuntimeError("\u274c FAILED: core dependency installation failed -- check the pip output above.")

# 4-bit extras -- same best-effort pattern as the repo's Dockerfile.
extras_status = os.system('pip install -q "bitsandbytes>=0.43.0" nvidia-nvjitlink-cu13')
if extras_status == 0:
    print("\u2705 SUCCESS: 4-bit quantization extras installed")
else:
    print("\u26a0\ufe0f  4-bit extras failed to install -- fine if GEMMA_USE_4BIT=0 (bf16 path).")

# Notebook-only tooling for calling the server and managing tunnels.
get_ipython().system("pip install -q requests")
print("\u2705 SUCCESS: Notebook tooling ready")


**Expected Output**
Two success messages for the core and 4-bit-extras installs (the second may warn and continue rather than fail), followed by confirmation that `requests` is available for the HTTP checks used throughout the rest of this notebook.

### Kernel restart

Restarts the kernel so `numpy` (and every package compiled against it — `torch`, `transformers`, `sentence-transformers`, `chromadb`) loads cleanly against the versions just installed above, exactly as `01_Setup_and_Retrieval_Validation.ipynb` and `02_Gemma_Inference_and_Serving.ipynb` both do after their own install step. **Colab will auto-reconnect in a few seconds — continue running from the next cell once it does.**

In [ ]:
print("Restarting the kernel so numpy loads cleanly after the install above...")
print("Colab will auto-reconnect in a few seconds -- then continue running from the next cell.")
import os
os.kill(os.getpid(), 9)


**Expected Output**
The restart message above, followed by an automatic Colab reconnect. After reconnecting, the working directory and every Python variable from before the restart are gone — the next cell re-establishes them before continuing.

### GPU detection

In [ ]:
import os

REPO_URL = "https://github.com/sunvantaconsultancysolutions-design/dmrc_deploy.git"
REPO_DIR = "/content/dmrc_deploy"
get_ipython().run_line_magic("cd", REPO_DIR)

import torch

GPU_AVAILABLE = torch.cuda.is_available()

if GPU_AVAILABLE:
    gpu_name = torch.cuda.get_device_name(0)
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"\u2705 GPU detected: {gpu_name} ({total_vram_gb:.1f} GB VRAM)")
    get_ipython().system("nvidia-smi --query-gpu=name,memory.total,memory.used,driver_version --format=csv")
else:
    total_vram_gb = 0.0
    print("\u26a0\ufe0f  No GPU detected. Gemma-2-9B-it, BGE-M3, and the reranker will run on CPU, "
          "which is extremely slow for production serving. Go to Runtime -> Change runtime type -> GPU and re-run.")

# Decide bf16 vs 4-bit for Gemma-2-9B-it from detected VRAM.
# bf16 needs ~18GB; below that, GEMMA_USE_4BIT=1 (read unchanged by
# src/gemma_inference.py) quantizes to fit a T4/L4. Matches the repo's own
# Dockerfile GEMMA_USE_4BIT knob -- chosen automatically here instead of hardcoded.
USE_4BIT = "0" if total_vram_gb >= 30 else "1"
print(f"\nGEMMA_USE_4BIT will be set to: {USE_4BIT} "
      f"({'bf16 full precision' if USE_4BIT == '0' else '4-bit quantized (bitsandbytes)'})")


**Expected Output**
GPU model name and VRAM (or a CPU warning), followed by the `GEMMA_USE_4BIT` decision this notebook will use for the rest of the run.

### Environment variables

Declares the production configuration surface up front, matching the current repository's `Dockerfile` defaults exactly (the same values validated in `02_Gemma_Inference_and_Serving.ipynb`'s FastAPI Integration section). These are applied to the server subprocess's environment in **Section 5**, not the notebook kernel's own `os.environ` — the server subprocess is a separate OS process.

In [ ]:
# Matches src/gemma_inference.py, src/retrieval_caps.py, and src/app.py's
# own env-var reads, at the exact values the repository's Dockerfile uses.
DEPLOYMENT_ENV = {
    "GEMMA_USE_4BIT": USE_4BIT,            # decided above from detected VRAM
    "GEMMA_MAX_NEW_TOKENS": "1536",        # Dockerfile default
    "RAG_MAX_CANDIDATES": "60",            # read by src/retrieval_caps.py
    "RAG_MAX_CONTEXT": "12",               # read by src/retrieval_caps.py
    "ALLOWED_ORIGINS": "*",                # CORS -- see src/app.py; tightened per-deploy in real prod
}

print("Deployment configuration for this session:")
for k, v in DEPLOYMENT_ENV.items():
    print(f"  {k:<22} = {v}")


## 3. Hugging Face Authentication

**Purpose**
Authenticate with Hugging Face so the gated `google/gemma-2-9b-it` checkpoint can be downloaded.

**Explanation**
`google/gemma-2-9b-it` is a **gated** checkpoint — the account logging in below must already have accepted its license at https://huggingface.co/google/gemma-2-9b-it, or model loading in Section 4 fails with a 401 / gated-repo error. `BAAI/bge-m3` and `BAAI/bge-reranker-v2-m3` are ungated and don't need this, but logging in once covers all three, matching `02_Gemma_Inference_and_Serving.ipynb`'s own authentication step.

Running the cell below will prompt for a Hugging Face access token (create one at https://huggingface.co/settings/tokens with at least *read* scope) — this is the one manual step this notebook requires.

In [ ]:
from huggingface_hub import login, whoami

login()  # paste a Hugging Face access token when prompted

try:
    user = whoami()
    print(f"\u2705 SUCCESS: Logged in to Hugging Face as '{user['name']}'")
except Exception as e:
    raise RuntimeError(f"\u274c FAILED: Hugging Face login could not be verified: {e}")


**Expected Output**
An interactive token-entry widget, followed by `✅ SUCCESS: Logged in to Hugging Face as '<username>'`.

**Verification** — the `whoami()` call above is the verification step: if the token is invalid or missing the required scope, it raises before any model download is attempted, failing fast rather than deep inside Section 4.

## 4. Model Preparation

**Purpose**
Load the existing production model — tokenizer, quantization config, and weights — exactly as `src/gemma_inference.py` already implements, smoke-test it, then release the notebook's own GPU memory before the FastAPI server (Section 5) loads its own copy.

**Explanation**
No training, fine-tuning, or model code is touched here. `src/gemma_inference.py`'s `get_gemma_model()` is the single source of truth for how the model is loaded — it:
- loads the **tokenizer** and **model** together, from the same `google/gemma-2-9b-it` checkpoint id;
- reads `GEMMA_USE_4BIT` from the environment to decide **quantization**: `0` loads full bf16 weights, `1` loads 4-bit NF4 weights via `bitsandbytes` (for smaller GPUs);
- places the model on **GPU** (`device_map="auto"`) with **memory optimization** (bf16 or 4-bit, not fp32) so it fits alongside the BGE-M3 and reranker models also resident during retrieval.

This cell sets `GEMMA_USE_4BIT` in the notebook's own process (not the server subprocess — that gets it independently in Section 5) purely so this smoke-test load uses the same quantization decision made in Section 2.

In [ ]:
import os
os.environ["GEMMA_USE_4BIT"] = USE_4BIT

from src.gemma_inference import get_gemma_model, generate_answer

model, tokenizer, device = get_gemma_model()
print(f"\u2705 Model loaded. device={device}  quantized={USE_4BIT == '1'}")

answer = generate_answer("What is Artificial Intelligence?")
print("\nSmoke-test generation:\n")
print(answer)


**Expected Output**
`✅ Model loaded. device=cuda ...` followed by a coherent smoke-test answer and a per-call throughput line (`prompt tok -> new tok in Ns (tok/s)`), confirming the tokenizer, quantization path, and GPU placement all work correctly before this notebook goes anywhere near the retrieval pipeline or the API.

### Memory optimization — release the notebook's GPU copy

The FastAPI server started in **Section 5** runs in a **separate OS process** and loads its own copy of Gemma-2-9B-it on the same GPU. Two resident bf16 Gemma copies (~18GB each) plus BGE-M3 and the reranker would risk out-of-memory on anything smaller than an 80GB A100 — so the notebook-side reference is cleared here, exactly as `02_Gemma_Inference_and_Serving.ipynb` does before starting its own server.

In [ ]:
import gc, torch

for _name in ("model", "tokenizer"):
    if _name in globals():
        del globals()[_name]

gc.collect()
torch.cuda.empty_cache()
print("\u2705 Notebook-side model reference cleared; GPU memory released for the server process.")


## 5. Backend Deployment

**Purpose**
Start the production FastAPI application (`src/app.py`) as a background `uvicorn` process, then verify it comes up healthy with every endpoint it exposes.

**Explanation**
Started **exactly** as the repository's own `Dockerfile` does: `uvicorn src.app:app --host 0.0.0.0 --port 8000`, run from the repo root, with the `DEPLOYMENT_ENV` values declared in Section 2 (`GEMMA_USE_4BIT`, `GEMMA_MAX_NEW_TOKENS`, `RAG_MAX_CANDIDATES`, `RAG_MAX_CONTEXT`, `ALLOWED_ORIGINS`). No line of `src/app.py` is edited — only its documented runtime configuration is supplied via environment variables, and `src/app.py` imports `src/retrieval_caps.py` directly at module load time, so the caps take effect the moment the server process starts.

In [ ]:
import subprocess, time, requests

server_env = {**os.environ, **DEPLOYMENT_ENV}

server_log_path = "/content/server.log"
server_log_file = open(server_log_path, "w")

server_process = subprocess.Popen(
    ["uvicorn", "src.app:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd=REPO_DIR,
    env=server_env,
    stdout=server_log_file,
    stderr=subprocess.STDOUT,
)

print(f"Server process started (pid={server_process.pid}). Logs -> {server_log_path}")
print("Polling GET /status until the server reports healthy "
      "(cold start with model downloads can take several minutes)...")


In [ ]:
# Poll GET /status (implemented in src/app.py) until the server responds.
MAX_WAIT_SECONDS = 1200  # generous: bf16 Gemma-2-9B cold start + BGE-M3 + reranker download/load
start = time.time()
server_up = False
status_json = None

while time.time() - start < MAX_WAIT_SECONDS:
    if server_process.poll() is not None:
        print("\u274c FAILED: server process exited early. Last log lines:")
        get_ipython().system(f"tail -n 80 {server_log_path}")
        break
    try:
        r = requests.get("http://127.0.0.1:8000/status", timeout=5)
        if r.status_code == 200:
            status_json = r.json()
            elapsed = time.time() - start
            print(f"\n\u2705 SUCCESS: Server is up after {elapsed:.0f}s")
            print(status_json)
            server_up = True
            break
    except requests.exceptions.RequestException:
        pass
    print(f"  ... still starting ({time.time() - start:.0f}s elapsed)", end="\r")
    time.sleep(10)

if not server_up:
    raise RuntimeError("\u274c FAILED: server did not become healthy within the wait window. "
                        "See Section 9 (Troubleshooting -> FastAPI startup failures).")


**Expected Output**
A polling loop ending with `✅ SUCCESS: Server is up after Ns`, followed by the parsed `/status` JSON, e.g.:

```json
{"status": "running", "embedding_model": "BAAI/bge-m3", "reranker_model": "BAAI/bge-reranker-v2-m3",
 "dense_model_loaded": true, "reranker_model_loaded": true, "gemma_model_loaded": true,
 "chromadb_connected": true}
```

### Health endpoint and required endpoints

`src/app.py` implements `GET /` as the lightweight health check and `GET /status` for detailed system status — there is no route literally named `/health` in this repository; both are checked below.

In [ ]:
def check_endpoint(name, method, path, base_url="http://127.0.0.1:8000", json_body=None, timeout=15):
    url = f"{base_url}{path}"
    try:
        if method == "GET":
            r = requests.get(url, timeout=timeout)
        else:
            r = requests.post(url, json=json_body, timeout=timeout)
        if r.status_code == 200:
            print(f"\u2705 SUCCESS: {name} ({method} {path}) -> {r.status_code}")
            return r.json()
        else:
            print(f"\u274c FAILED: {name} ({method} {path}) -> HTTP {r.status_code}: {r.text}")
            return None
    except requests.exceptions.RequestException as e:
        print(f"\u274c FAILED: {name} ({method} {path}) -> {e}")
        return None

print("--- Required endpoints (local, pre-tunnel) ---")
health = check_endpoint("Health check", "GET", "/")
status = check_endpoint("Status", "GET", "/status")
print()
print("Health response :", health)
print("Status response :", status)


**Expected Output**
Two `✅ SUCCESS` lines with the health and status payloads printed underneath.

### Logging

The full server output — including every request handled — is captured continuously in `/content/server.log` (opened in the startup cell above). Tail it any time to inspect request-level activity or diagnose an error without stopping the server.

In [ ]:
get_ipython().system(f"tail -n 30 {server_log_path}")


## 6. Cloudflare Tunnel

**Purpose**
Expose the local backend (`127.0.0.1:8000`) on a public HTTPS URL using a free, no-account `trycloudflare.com` Quick Tunnel — no changes to the repository, just a reverse proxy in front of the already-running FastAPI process.

**Explanation**
Quick Tunnels are ephemeral: the URL is generated fresh each time the tunnel starts and stays valid only while `tunnel_process` (below) keeps running in this Colab session.

In [ ]:
# Installation — download the cloudflared binary (Linux amd64, matches the Colab runtime).
get_ipython().system(
    "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 "
    "-O /content/cloudflared && chmod +x /content/cloudflared"
)

if os.path.exists("/content/cloudflared") and os.access("/content/cloudflared", os.X_OK):
    print("\u2705 SUCCESS: cloudflared binary downloaded")
else:
    raise RuntimeError("\u274c FAILED: could not download cloudflared. "
                        "See Section 9 (Troubleshooting -> Cloudflare Tunnel issues) for a fallback.")


In [ ]:
import re

def start_quick_tunnel(local_port, log_path):
    """Start a cloudflared Quick Tunnel and return (process, public_url)."""
    log_file = open(log_path, "w")
    proc = subprocess.Popen(
        ["/content/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{local_port}"],
        stdout=log_file,
        stderr=subprocess.STDOUT,
    )
    url_pattern = re.compile(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com")
    public_url = None
    start = time.time()
    while time.time() - start < 60:
        if os.path.exists(log_path):
            with open(log_path) as f:
                contents = f.read()
            match = url_pattern.search(contents)
            if match:
                public_url = match.group(0)
                break
        time.sleep(1)
    return proc, public_url

# Tunnel startup for the backend.
tunnel_process, BACKEND_PUBLIC_URL = start_quick_tunnel(8000, "/content/cloudflared_backend.log")

if BACKEND_PUBLIC_URL:
    print(f"\u2705 SUCCESS: Backend public URL -- {BACKEND_PUBLIC_URL}")
else:
    print("\u274c FAILED: Cloudflare did not return a public URL within 60s. "
          "See Section 9 (Troubleshooting -> Cloudflare Tunnel issues) for the ngrok fallback.")


**Expected Output**
`✅ SUCCESS: Backend public URL -- https://<random-words>.trycloudflare.com`

### Verification

Re-runs the same health checks as Section 5, but through the public tunnel URL this time — confirming the backend is reachable from *outside* this Colab VM, not just from inside it (which is what actually matters for a live demo).

In [ ]:
print("--- Required endpoints (public, via Cloudflare Tunnel) ---")
public_health = check_endpoint("Health check", "GET", "/", base_url=BACKEND_PUBLIC_URL)
public_status = check_endpoint("Status", "GET", "/status", base_url=BACKEND_PUBLIC_URL)


**Fallback: ngrok** — only needed if the Cloudflare cell above did not return a public URL (see Section 9). Requires a free ngrok account and authtoken (https://dashboard.ngrok.com/get-started/your-authtoken).

In [ ]:
# --- ngrok fallback, only run this if the Cloudflare Tunnel above failed ---
#
# get_ipython().system("pip install -q pyngrok")
# from pyngrok import ngrok
# NGROK_AUTHTOKEN = ""  # <-- paste your token here
# ngrok.set_auth_token(NGROK_AUTHTOKEN)
# ngrok_tunnel = ngrok.connect(8000)
# BACKEND_PUBLIC_URL = ngrok_tunnel.public_url
# print(f"\u2705 SUCCESS: Backend public URL (ngrok) -- {BACKEND_PUBLIC_URL}")


## 7. End-to-End Testing

**Purpose**
Demonstrate the complete, live, publicly-hosted backend pipeline with real DMRC contract queries, exercising the same path any client (including a locally-run React frontend, outside this notebook) would trigger:

```
Client Request  (POST /ask, tunneled)
    ↓
FastAPI
    ↓
Hybrid Retrieval  (dense BGE-M3 + BM25)
    ↓
Reranker  (BGE-reranker-v2-m3, cross-encoder)
    ↓
Gemma-2-9B-it  (grounded generation)
    ↓
Answer
    ↓
Source Citations
    ↓
Confidence Score
```

This section calls `POST /ask` on `BACKEND_PUBLIC_URL` directly — the same endpoint a locally-run frontend would call — so these results are exactly what any client of the deployed backend would see.

In [ ]:
SAMPLE_QUERIES = [
    "What are the contractor obligations?",
    "Who is responsible for maintenance during the defects liability period?",
    "Explain clause 1.2.1",
    "What is the scope of work?",
    "What is the quantity and rate for the cooling tower BOQ item?",  # BOQ-grounded query
]

print("Warming up (first call includes any remaining lazy initialization)...")
t0 = time.time()
try:
    warm = requests.post(f"{BACKEND_PUBLIC_URL}/ask", json={"query": "warmup"}, timeout=600)
    print(f"Warm-up done in {time.time()-t0:.0f}s -- status {warm.status_code}")
except requests.exceptions.RequestException as e:
    print(f"Warm-up failed after {time.time()-t0:.0f}s: {e}")


In [ ]:
results = []

for q in SAMPLE_QUERIES:
    print("=" * 78)
    print(f"QUERY: {q}")
    t0 = time.time()
    try:
        r = requests.post(f"{BACKEND_PUBLIC_URL}/ask", json={"query": q}, timeout=300)
        elapsed = time.time() - t0
        if r.status_code == 200:
            body = r.json()
            answer = body.get("answer", "")
            sources = body.get("sources", [])
            confidence = body.get("confidence")
            print(f"  status      : \u2705 200 OK ({elapsed:.1f}s)")
            print(f"  confidence  : {confidence}")
            print(f"  sources     : {len(sources)}")
            if sources:
                first = sources[0]
                print(f"  top source  : {first}")
            print(f"  answer      : {answer[:400]}{'...' if len(answer) > 400 else ''}")
            results.append({"query": q, "status": 200, "elapsed_s": elapsed,
                             "confidence": confidence, "n_sources": len(sources)})
        else:
            print(f"  status      : \u274c HTTP {r.status_code}: {r.text[:200]}")
            results.append({"query": q, "status": r.status_code, "elapsed_s": elapsed,
                             "confidence": None, "n_sources": 0})
    except requests.exceptions.RequestException as e:
        print(f"  status      : \u274c request failed: {e}")
        results.append({"query": q, "status": "error", "elapsed_s": None,
                         "confidence": None, "n_sources": 0})
    print()

print("=" * 78)
print("SUMMARY")
for row in results:
    print(f"  [{row['status']}]  {row['elapsed_s']}s  conf={row['confidence']}  "
          f"sources={row['n_sources']}  -- {row['query']}")


**Expected Output**
For each query: HTTP status, elapsed time, confidence score, source count, the top source's metadata, and a preview of the generated answer with inline citations (e.g. *"Clause 4.2, Page 4"*). The BOQ query is expected to resolve to `chunk_type="boq"` sources carrying `s_no` rather than `clause_no`. A final summary table lists every query's status side by side.

## 8. Deployment Validation

**Purpose**
Run a single consolidated checklist over the results already gathered above, confirming every layer of the deployed system is functioning correctly with no runtime errors.

In [ ]:
checks = {}

# 1. API works
checks["API responds (health + status)"] = bool(public_health and public_status)

# 2. Model works -- Gemma reported loaded by /status
checks["Gemma model loaded"] = bool(public_status and public_status.get("gemma_model_loaded"))

# 3. Retrieval works -- dense + reranker reported loaded
checks["Dense retrieval model loaded"] = bool(public_status and public_status.get("dense_model_loaded"))
checks["Reranker model loaded"] = bool(public_status and public_status.get("reranker_model_loaded"))
checks["ChromaDB connected"] = bool(public_status and public_status.get("chromadb_connected"))

# 4. Citations work -- every successful /ask call in Section 7 returned >= 1 source
non_boq_results = [r for r in results if r["query"] != "What is the quantity and rate for the cooling tower BOQ item?"]
checks["Citations returned for grounded queries"] = all(
    r["status"] == 200 and r["n_sources"] > 0 for r in non_boq_results
)

# 5. Confidence score works -- every successful call returned a numeric confidence
checks["Confidence score present"] = all(
    r["status"] == 200 and r["confidence"] is not None for r in results
)

# 6. No runtime errors -- every query in Section 7 returned HTTP 200
checks["No runtime errors across sample queries"] = all(r["status"] == 200 for r in results)

print("=" * 60)
print("  DEPLOYMENT VALIDATION CHECKLIST")
print("=" * 60)
all_passed = True
for name, passed in checks.items():
    mark = "\u2705 PASS" if passed else "\u274c FAIL"
    if not passed:
        all_passed = False
    print(f"  {mark}  {name}")
print("=" * 60)
print("OVERALL: \u2705 ALL CHECKS PASSED" if all_passed else "OVERALL: \u274c ONE OR MORE CHECKS FAILED -- see Section 9")


**Expected Output**
A checklist with every item marked `✅ PASS`, ending in `OVERALL: ✅ ALL CHECKS PASSED`. Any `❌ FAIL` line points directly at which subsystem to investigate in the troubleshooting table below.

## 9. Troubleshooting

No code runs in this section — it is a reference table for the failure modes most likely to occur when re-running this notebook in a fresh Colab session.

| Problem | Symptom | Solution |
|---|---|---|
| **Hugging Face authentication issues** | `401`/`403` "gated repo" error when loading `google/gemma-2-9b-it` in Section 4 | Confirm the token pasted into `login()` belongs to an account that has accepted the license at https://huggingface.co/google/gemma-2-9b-it. Re-run the Section 3 cell with a fresh token from https://huggingface.co/settings/tokens (read scope is sufficient). |
| **GPU memory (VRAM) issues** | `CUDA out of memory` during model load or generation | Set `GEMMA_USE_4BIT=1` (Section 2 already does this automatically below ~30GB VRAM). Confirm Section 4's memory-optimization cell ran (`torch.cuda.empty_cache()`) before starting the server in Section 5 — two resident bf16 Gemma copies will exhaust anything smaller than an 80GB A100. Restart the runtime (`Runtime → Restart runtime`) to fully clear VRAM if a previous run left it fragmented. |
| **Package installation failures** | `pip install` errors in Section 2, especially around `bitsandbytes` / `nvidia-nvjitlink-cu13` | These two packages are installed *best-effort* and are only required for the 4-bit path — a failure here is non-fatal if running bf16 (`GEMMA_USE_4BIT=0`). For failures in the core `requirements.txt` install, re-run the cell (transient PyPI/network issues are the most common cause) or check Colab's own pre-installed package versions for a conflict. |
| **Cloudflare Tunnel issues** | Section 6 tunnel cell doesn't return a `trycloudflare.com` URL within 60s | Re-run the tunnel-start cell — `trycloudflare.com` Quick Tunnels occasionally fail to register on first attempt. If it keeps failing, use the ngrok fallback cell provided directly below the tunnel cell (requires a free ngrok account + authtoken). Check `/content/cloudflared_backend.log` directly for the underlying error. |
| **FastAPI startup failures** | Section 5's polling loop times out, or `server_process.poll()` shows the process exited | Check `/content/server.log` (printed automatically on failure by the polling cell) for the actual traceback — most commonly a missing `chroma_db/` (confirm the clone in Section 2 completed fully) or a model-loading error already covered above. Confirm you are running from the repo root (`REPO_DIR`), since `src/app.py`'s relative imports depend on it. |
| **Local frontend can't reach the backend** | A locally-run React app (`frontend/`, outside this notebook) shows a network/CORS error when asking a question | Set the locally-run frontend's `VITE_API_URL` to the *current* `BACKEND_PUBLIC_URL` printed in Section 6 and again in Section 10's summary (Quick Tunnel URLs change every time the tunnel restarts). Confirm `ALLOWED_ORIGINS=*` is set in `DEPLOYMENT_ENV` (Section 2) so the backend's CORS middleware accepts requests from the local frontend's origin. |

**General tip:** every long-running process in this notebook (`server_process`, `tunnel_process`) logs to a file in `/content/`. When something looks wrong, `tail` the relevant log before re-running cells — it is almost always faster than guessing.

## 10. Conclusion

This notebook deployed the backend of the completed DMRC Contract Intelligence RAG system — from a clean Colab runtime to a publicly reachable FastAPI API — reusing the `dmrc_deploy` repository exactly as implemented, with no changes to retrieval, reranking, prompt engineering, or generation logic. The React frontend runs locally, outside this notebook, and simply points its `VITE_API_URL` at the backend URL produced below.

In [ ]:
print("=" * 70)
print("  DMRC CONTRACT INTELLIGENCE -- BACKEND DEPLOYMENT SUMMARY")
print("=" * 70)
print(f"  Backend  (FastAPI, tunneled) : {BACKEND_PUBLIC_URL}")
print(f"    Health check                : {BACKEND_PUBLIC_URL}/")
print(f"    Status                      : {BACKEND_PUBLIC_URL}/status")
print(f"    Ask endpoint                : {BACKEND_PUBLIC_URL}/ask  (POST, JSON: {{\"query\": \"...\"}})")
print("=" * 70)
print(f"  Successful deployment  : {'YES' if all_passed else 'PARTIAL -- see Section 9'}")
print(f"  Public API available   : YES")
print(f"  End-to-end functionality: {len([r for r in results if r['status'] == 200])}/{len(results)} sample queries succeeded")
print("=" * 70)
print()
print(f"For a locally-run React frontend, set VITE_API_URL={BACKEND_PUBLIC_URL}")
print()
print("\u26a0\ufe0f  This public URL is only valid while this Colab session keeps running --")
print("   re-running Sections 5-6 in a new session will generate a new URL.")


**Summary**
- **Successful deployment** — the backend (FastAPI + retrieval + Gemma-2-9B-it) started cleanly from the unmodified `dmrc_deploy` repository and passed every check in Section 8's validation checklist.
- **Public API availability** — `BACKEND_PUBLIC_URL` serves `GET /`, `GET /status`, and `POST /ask` over HTTPS via Cloudflare Tunnel, reachable from outside this Colab VM.
- **End-to-end functionality** — Section 7 demonstrated the complete pipeline (hybrid retrieval → reranking → Gemma generation → cited, confidence-scored answers) against both free-text and BOQ contract queries, exactly as any client of the deployed backend would experience it.

This backend is now ready for a locally-run frontend, or any other HTTP client, to connect to via `BACKEND_PUBLIC_URL`. For a persistent (non-Colab) production deployment, see the repository's `Dockerfile` and `README.md`, which this notebook's environment variables and startup commands were kept in exact sync with throughout.

---

### Optional cleanup

Stops every background process started in this notebook (backend server, backend tunnel) without ending the Colab session — useful before re-running Sections 5-6 with different configuration.

In [ ]:
for proc, label in [
    (server_process, "backend server"),
    (tunnel_process, "backend tunnel"),
]:
    try:
        proc.terminate()
        proc.wait(timeout=15)
        print(f"\u2705 {label} stopped.")
    except Exception:
        try:
            proc.kill()
            print(f"\u2705 {label} force-stopped.")
        except Exception as e:
            print(f"\u26a0\ufe0f  Could not stop {label}: {e}")
